# no-grad-context-mgr-update — worked example 3: NoGrad restores even when the block raises

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `no-grad-context-mgr-update`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper.
    `.grad` accumulates the leaf gradient at the end of the reverse pass."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

`__exit__` always fires, even on the exception path, so a save-restore `NoGrad` returns the flag to its previous value whether the block completed normally or raised. Returning a falsy value from `__exit__` lets the exception propagate.

## Worked solution

We use the same `NoGrad` and deliberately raise inside the block. Because `__exit__` is part of the context-manager contract, it runs during stack unwinding and restores `grad_tracking_enabled` to its saved previous value. We do NOT return a truthy value from `__exit__`, so the exception is re-raised after restoration. We wrap the whole thing in a `try/except`, confirm the exception escaped, and print that the flag is back to `True` afterward, proving cleanup happened despite the error.

In [ ]:
grad_tracking_enabled = True

class NoGrad:
    def __enter__(self):
        global grad_tracking_enabled
        self._prev = grad_tracking_enabled
        grad_tracking_enabled = False
        return self
    def __exit__(self, exc_type, exc_val, exc_tb):
        global grad_tracking_enabled
        grad_tracking_enabled = self._prev

raised = False
try:
    with NoGrad():
        raise ValueError('boom')
except ValueError:
    raised = True
print('exception propagated:', raised)
print('flag restored:', grad_tracking_enabled)